# Phase 8 — RFM Customer Segmentation

### Goal

segments Olist customers using RFM analysis:

- **Recency**: How recently a customer purchased
- **Frequency**: How many times a customer purchased
- **Monetary**: How much revenue the customer generated

The business goal is to translate transaction data into actionable customer groups that a marketing or CRM team could use for targeted campaigns.

Examples:

- Champions: recent, frequent, high-value customers
- At Risk: valuable customers who have not purchased recently
- Lost: inactive, low-value customers
- Potential Loyalists: recent customers who could become loyal with the right follow-up


In [ ]:
# ============================================================
# OLIST RFM CUSTOMER SEGMENTATION
# Phase 8 — Customer Analytics
# ============================================================

from pathlib import Path
import os

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns

from dotenv import load_dotenv
from sqlalchemy import create_engine
from sqlalchemy.engine import URL

# ------------------------------------------------------------
# Project paths
# ------------------------------------------------------------

PROJECT_ROOT = Path.cwd().parent
ENV_PATH = PROJECT_ROOT / ".env"

FIGURES_DIR = PROJECT_ROOT / "outputs" / "figures"
EXPORTS_DIR = PROJECT_ROOT / "outputs" / "exports"

FIGURES_DIR.mkdir(parents=True, exist_ok=True)
EXPORTS_DIR.mkdir(parents=True, exist_ok=True)

# ------------------------------------------------------------
# Load database credentials
# ------------------------------------------------------------

load_dotenv(ENV_PATH, override=True)

db_url = URL.create(
    drivername="postgresql+psycopg2",
    username=os.getenv("DB_USER"),
    password=os.getenv("DB_PASSWORD"),
    host=os.getenv("DB_HOST"),
    port=int(os.getenv("DB_PORT")),
    database=os.getenv("DB_NAME"),
)

engine = create_engine(db_url)

print("Connected successfully")
print("Project root:", PROJECT_ROOT)

In [ ]:
# Clean, portfolio-friendly chart style

sns.set_theme(style="whitegrid")

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.family": "sans-serif",
    "font.size": 11,
})

def save_fig(filename):
    """
    Save the current matplotlib figure into outputs/figures.
    """
    path = FIGURES_DIR / filename
    plt.tight_layout()
    plt.savefig(path, dpi=150, bbox_inches="tight")
    plt.show()
    print(f"Saved figure: {path}")

## Step 1 — Pull customer purchase data from PostgreSQL

For RFM analysis, need customer-level purchase behavior.

Important:

Use `customer_unique_id`, not `customer_id`.

In the Olist dataset, `customer_id` is order-level, while `customer_unique_id` represents the real customer across multiple orders. Using `customer_id` would make repeat purchase and customer lifetime value analysis incorrect.


In [ ]:
query = """
SELECT
    c.customer_unique_id,
    o.order_id,
    o.order_purchase_timestamp,
    SUM(oi.price + oi.freight_value) AS order_value
FROM orders o
JOIN customers c
    ON o.customer_id = c.customer_id
JOIN order_items oi
    ON o.order_id = oi.order_id
WHERE o.order_status = 'delivered'
  AND o.order_purchase_timestamp >= '2017-01-01'
  AND o.order_purchase_timestamp < '2018-09-01'
GROUP BY
    c.customer_unique_id,
    o.order_id,
    o.order_purchase_timestamp
ORDER BY
    c.customer_unique_id,
    o.order_purchase_timestamp;
"""

orders_df = pd.read_sql(query, engine)

orders_df["order_purchase_timestamp"] = pd.to_datetime(
    orders_df["order_purchase_timestamp"]
)

orders_df.head()

In [ ]:
print("Rows loaded:", len(orders_df))
print("Unique customers:", orders_df["customer_unique_id"].nunique())
print("Unique orders:", orders_df["order_id"].nunique())

print("\nDate range:")
print("Min purchase date:", orders_df["order_purchase_timestamp"].min())
print("Max purchase date:", orders_df["order_purchase_timestamp"].max())

print("\nMissing values:")
print(orders_df.isna().sum())

print("\nPreview:")
display(orders_df.head())

## Step 2 — Calculate RFM metrics

For each customer:

- **Last purchase date** = most recent order date
- **Recency** = number of days since last purchase
- **Frequency** = number of delivered orders
- **Monetary** = total customer revenue

The analysis date is set as one day after the latest purchase date in the dataset. This avoids using today's date, because the dataset is historical.

In [ ]:
# Use one day after the dataset's latest order date as the reference date
analysis_date = orders_df["order_purchase_timestamp"].max() + pd.Timedelta(days=1)

rfm = (
    orders_df.groupby("customer_unique_id")
    .agg(
        last_purchase_date=("order_purchase_timestamp", "max"),
        frequency=("order_id", "nunique"),
        monetary=("order_value", "sum"),
    )
    .reset_index()
)

rfm["recency"] = (analysis_date - rfm["last_purchase_date"]).dt.days

# Reorder columns
rfm = rfm[
    [
        "customer_unique_id",
        "last_purchase_date",
        "recency",
        "frequency",
        "monetary",
    ]
]

print("Analysis date:", analysis_date.date())
display(rfm.head())

In [ ]:
print("RFM summary statistics:")
display(rfm[["recency", "frequency", "monetary"]].describe().round(2))

print("\nFrequency distribution:")
display(rfm["frequency"].value_counts().sort_index().head(20))

## Step 3 — Create RFM scores

Each customer receives a score for Recency, Frequency, and Monetary value.

Scoring logic:

- **R_score**: lower recency is better, because the customer purchased more recently
- **F_score**: higher frequency is better
- **M_score**: higher monetary value is better

Important note:

Olist has very low repeat purchasing behavior, so most customers purchased only once. Because of that, frequency cannot be split cleanly into 5 equal groups. Instead of forcing artificial frequency quintiles, this notebook uses business-based frequency scoring.

In [ ]:
# ------------------------------------------------------------
# Recency score
# Lower recency = better customer = higher score
# ------------------------------------------------------------

rfm["R_score"] = pd.qcut(
    rfm["recency"],
    q=5,
    labels=[5, 4, 3, 2, 1]
).astype(int)

# ------------------------------------------------------------
# Monetary score
# Higher monetary value = better customer = higher score
# ------------------------------------------------------------

rfm["M_score"] = pd.qcut(
    rfm["monetary"],
    q=5,
    labels=[1, 2, 3, 4, 5]
).astype(int)

# ------------------------------------------------------------
# Frequency score
# Most Olist customers only purchased once.
# So we use business thresholds instead of qcut.
# ------------------------------------------------------------

def score_frequency(freq):
    if freq == 1:
        return 1
    elif freq == 2:
        return 3
    elif freq == 3:
        return 4
    else:
        return 5

rfm["F_score"] = rfm["frequency"].apply(score_frequency)

# Combined RFM score
rfm["rfm_score"] = (
    rfm["R_score"].astype(str)
    + rfm["F_score"].astype(str)
    + rfm["M_score"].astype(str)
)

rfm["rfm_combined"] = rfm["R_score"] + rfm["F_score"] + rfm["M_score"]

display(rfm.head())

In [ ]:
print("R score distribution:")
display(rfm["R_score"].value_counts().sort_index())

print("\nF score distribution:")
display(rfm["F_score"].value_counts().sort_index())

print("\nM score distribution:")
display(rfm["M_score"].value_counts().sort_index())

print("\nCombined RFM score summary:")
display(rfm["rfm_combined"].describe().round(2))

## Step 4 — Assign business customer segments

The goal is not only to score customers, but to turn the scores into business-friendly segments.

These segments are easier for non-technical stakeholders to understand:

- **Champions**: recent, frequent, high-spending customers
- **Loyal Customers**: repeat customers with good overall value
- **Potential Loyalists**: recent customers who may become loyal
- **Recent Customers**: new or recent customers with low frequency so far
- **Promising**: recent customers with some potential but low value/frequency
- **Needs Attention**: middle-priority customers
- **At Risk**: repeat or valuable customers who have not purchased recently
- **Can't Lose Them**: high-value customers who are becoming inactive
- **Lost**: old, low-frequency, low-value customers

In [ ]:
def assign_segment(row):
    r = row["R_score"]
    f = row["F_score"]
    m = row["M_score"]

    if r >= 4 and f >= 3 and m >= 4:
        return "Champions"

    elif r >= 3 and f >= 3:
        return "Loyal Customers"

    elif r >= 4 and f == 1 and m >= 3:
        return "Potential Loyalists"

    elif r >= 4 and f == 1:
        return "Recent Customers"

    elif r == 3 and f == 1 and m >= 3:
        return "Promising"

    elif r <= 2 and f >= 3 and m >= 3:
        return "Can't Lose Them"

    elif r <= 2 and (f >= 3 or m >= 4):
        return "At Risk"

    elif r == 1 and f == 1 and m <= 2:
        return "Lost"

    else:
        return "Needs Attention"


rfm["segment"] = rfm.apply(assign_segment, axis=1)

display(rfm.head())

In [ ]:
segment_summary = (
    rfm.groupby("segment")
    .agg(
        customer_count=("customer_unique_id", "count"),
        avg_recency=("recency", "mean"),
        avg_frequency=("frequency", "mean"),
        avg_monetary=("monetary", "mean"),
        total_revenue=("monetary", "sum"),
    )
    .reset_index()
)

segment_summary["customer_pct"] = (
    segment_summary["customer_count"] / segment_summary["customer_count"].sum() * 100
)

segment_summary["revenue_pct"] = (
    segment_summary["total_revenue"] / segment_summary["total_revenue"].sum() * 100
)

segment_summary = segment_summary.sort_values(
    "total_revenue",
    ascending=False
)

display(segment_summary.round(2))

## Step 5 — Visualize customer count by segment

This chart shows the size of each customer segment.

Business use:

A marketing team can use this to understand whether the customer base is mostly active, inactive, high-value, or low-value.

In [ ]:
segment_order_count = segment_summary.sort_values("customer_count", ascending=True)

plt.figure(figsize=(11, 6))

ax = sns.barplot(
    data=segment_order_count,
    x="customer_count",
    y="segment",
    hue="segment",
    dodge=False,
    legend=False,
    palette="viridis"
)

ax.set_title("Customer Count by RFM Segment", fontsize=14, pad=15)
ax.set_xlabel("Number of Customers")
ax.set_ylabel("Customer Segment")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"{x:,.0f}"))

for container in ax.containers:
    ax.bar_label(container, fmt="%.0f", padding=3)

save_fig("07_customer_count_by_rfm_segment.png")

## Step 6 — Visualize revenue by segment

This chart shows which customer segments contribute the most revenue.

Business use:

A segment may contain many customers but contribute little revenue, or contain fewer customers but generate high revenue. This helps prioritize marketing resources.

In [ ]:
segment_order_revenue = segment_summary.sort_values("total_revenue", ascending=True)

plt.figure(figsize=(11, 6))

ax = sns.barplot(
    data=segment_order_revenue,
    x="total_revenue",
    y="segment",
    hue="segment",
    dodge=False,
    legend=False,
    palette="magma"
)

ax.set_title("Revenue by RFM Segment", fontsize=14, pad=15)
ax.set_xlabel("Total Revenue")
ax.set_ylabel("Customer Segment")
ax.xaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"R${x/1e6:.1f}M"))

for container in ax.containers:
    ax.bar_label(
        container,
        labels=[f"R${v/1e6:.2f}M" for v in segment_order_revenue["total_revenue"]],
        padding=3
    )

save_fig("08_revenue_by_rfm_segment.png")

## Step 7 — Recency vs Monetary scatter plot

This scatter plot compares how recently customers purchased against how much they spent.

To make the chart readable, it removes extreme outliers by showing only customers below the 99th percentile of monetary value.

Business use:

This helps identify:

- High-value recent customers
- High-value inactive customers
- Low-value lost customers
- Recent but low-spending customers with growth potential

In [ ]:
plot_df = rfm[
    (rfm["monetary"] <= rfm["monetary"].quantile(0.99))
].copy()

plt.figure(figsize=(12, 7))

ax = sns.scatterplot(
    data=plot_df,
    x="recency",
    y="monetary",
    hue="segment",
    alpha=0.55,
    s=35
)

ax.set_title(
    "RFM Segments: Recency vs Monetary Value",
    fontsize=14,
    pad=15
)

ax.set_xlabel("Recency: Days Since Last Purchase")
ax.set_ylabel("Monetary Value: Customer Lifetime Revenue")
ax.yaxis.set_major_formatter(mtick.FuncFormatter(lambda x, _: f"R${x:,.0f}"))

plt.legend(
    title="Segment",
    bbox_to_anchor=(1.05, 1),
    loc="upper left"
)

save_fig("09_rfm_recency_vs_monetary_scatter.png")

## Step 8 — Export RFM results

The customer-level RFM table and segment summary are exported as CSV files.

These files can later be imported into Power BI for the dashboard phase.

In [ ]:
rfm_export_cols = [
    "customer_unique_id",
    "last_purchase_date",
    "recency",
    "frequency",
    "monetary",
    "R_score",
    "F_score",
    "M_score",
    "rfm_score",
    "rfm_combined",
    "segment",
]

rfm_export = rfm[rfm_export_cols].copy()

rfm_export.to_csv(EXPORTS_DIR / "rfm_segments.csv", index=False)
segment_summary.to_csv(EXPORTS_DIR / "rfm_segment_summary.csv", index=False)

print("Exported files:")
print(EXPORTS_DIR / "rfm_segments.csv")
print(EXPORTS_DIR / "rfm_segment_summary.csv")

In [ ]:
total_customers = len(rfm)
total_revenue = rfm["monetary"].sum()

top_segment_by_customers = segment_summary.sort_values(
    "customer_count",
    ascending=False
).iloc[0]

top_segment_by_revenue = segment_summary.sort_values(
    "total_revenue",
    ascending=False
).iloc[0]

champions = segment_summary[
    segment_summary["segment"] == "Champions"
]

at_risk = segment_summary[
    segment_summary["segment"].isin(["At Risk", "Can't Lose Them"])
]

print("RFM CUSTOMER SEGMENTATION INSIGHTS")
print("=" * 45)

print(f"Total customers analyzed: {total_customers:,.0f}")
print(f"Total delivered revenue analyzed: R${total_revenue:,.2f}")

print()
print(
    f"The largest customer segment by customer count is "
    f"'{top_segment_by_customers['segment']}', with "
    f"{top_segment_by_customers['customer_count']:,.0f} customers "
    f"({top_segment_by_customers['customer_pct']:.2f}% of customers)."
)

print(
    f"The largest customer segment by revenue is "
    f"'{top_segment_by_revenue['segment']}', generating "
    f"R${top_segment_by_revenue['total_revenue']:,.2f} "
    f"({top_segment_by_revenue['revenue_pct']:.2f}% of revenue)."
)

if len(champions) > 0:
    champions_row = champions.iloc[0]
    print()
    print(
        f"Champions represent {champions_row['customer_pct']:.2f}% of customers "
        f"and generate {champions_row['revenue_pct']:.2f}% of revenue. "
        "These customers are strong candidates for loyalty rewards, early access campaigns, "
        "and referral programs."
    )

if len(at_risk) > 0:
    at_risk_customers = at_risk["customer_count"].sum()
    at_risk_revenue = at_risk["total_revenue"].sum()

    print()
    print(
        f"At-risk high-value segments contain {at_risk_customers:,.0f} customers "
        f"and represent R${at_risk_revenue:,.2f} in historical revenue. "
        "These customers are good targets for win-back campaigns, personalized discounts, "
        "or reactivation emails."
    )

print()
print(
    "Overall interpretation: Because Olist behaves like a marketplace, many customers purchase only once. "
    "RFM segmentation helps separate low-priority one-time buyers from recent high-potential customers "
    "and inactive high-value customers who may be worth reactivating."
)

#excepted file structure created

# outputs/figures/
# ├── 07_customer_count_by_rfm_segment.png
# ├── 08_revenue_by_rfm_segment.png
# └── 09_rfm_recency_vs_monetary_scatter.png

# outputs/exports/
# ├── rfm_segments.csv
# └── rfm_segment_summary.csv

## Business Interpretation

RFM segmentation turns raw transaction history into customer groups that are easier for business teams to act on.

The key idea is that not all customers should receive the same marketing strategy.

Examples:

- **Champions** should receive loyalty rewards, early access, or referral campaigns.
- **Potential Loyalists** should receive personalized recommendations to encourage a second purchase.
- **Recent Customers** should receive onboarding or follow-up campaigns.
- **At Risk** and **Can't Lose Them** customers should receive win-back campaigns because they have historical value but have not purchased recently.
- **Lost** customers should receive low-cost reactivation campaigns, or may be deprioritized if marketing budget is limited.

This analysis is especially useful for the next dashboard phase because segment-level metrics can be shown directly in Power BI.